# Notebook 02 v2 — Corrected Transformer Fine-Tuning

**Project:** Benchmarking Multilingual Transformers for Malay-English Code-Switched Sentiment Analysis  
**Corpus:** MESocSentiment  
**Models:** XLM-R, mDeBERTa-v3, XLM-T

---

## What was wrong in v1 (`02_transformer_finetuning.ipynb`) and what is fixed here

| # | Flaw | Impact | Fix applied in v2 |
|---|------|--------|-------------------|
| 1 | **Noisy dev set** — checkpoint selection used 991 auto-labelled samples (~33.8% noise rate) | Model memorised labelling errors; best checkpoint selected on corrupted signal | Use `CLEAN_DEV_CSV` (300 gold-annotated samples) for checkpoint selection |
| 2 | **Only 3 epochs** — insufficient for a 278M-parameter model adapting from formal text to code-switched Twitter | Model never fully converged; weights underfit the training distribution | Increased to 7 epochs (`FINETUNE_CONFIG_V2`) |
| 3 | **mDeBERTa BF16 bug** — `torch.cuda.is_bf16_supported()` returns `True` on RTX 4060 Ti (Ada Lovelace), so the guard `bf16 = not torch.cuda.is_bf16_supported()` silently enabled BF16 | mDeBERTa ContextPooler produces extreme logits at init → NaN gradients → GradScaler skips all optimizer steps → **zero weight updates across all epochs** | Force `fp16=False, bf16=False` unconditionally for mDeBERTa; also re-initialise the classifier head |
| 4 | **No gradient clipping** | mDeBERTa prone to gradient explosions with stacked linear layers | `max_grad_norm=1.0` via `FINETUNE_CONFIG_V2` |

**Additional improvements (not in v1):**
- Differential learning rates: classification head at `2e-5`, backbone at `5e-6` (via `DifferentialLRTrainer`)
- Evaluation on clean 1,700-sample test set (not the full 2,000 which includes the 300 used for dev)

---

## Reproducibility note
The training set remains the same noisy auto-labelled set (unavoidable — only the gold test set has human labels).  
The dev/test split is created in `01_data_pipeline_and_svm.ipynb` and saved to `CLEAN_DEV_CSV` / `CLEAN_TEST_CSV`.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["HF_TOKEN"] = ""  # REMOVED: set HF_TOKEN env var before running

import sys
import sys
from pathlib import Path
_cwd = Path.cwd()
for _p in [_cwd / '../src', _cwd / '../3_Source', _cwd / 'src', _cwd / '3_Source']:
    if (_p / 'config.py').exists():
        sys.path.insert(0, str(_p.resolve()))
        break
else:
    raise ImportError('config.py not found -- run Jupyter from the project root or notebooks/ directory')

import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from config import (
    SEED, LABEL2ID, ID2LABEL, NUM_LABELS,
    RESULTS_DIR, MODELS_V2, FINETUNE_CONFIG_V2,
    CLEAN_DEV_CSV, CLEAN_TEST_CSV
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

BATCH_SIZE = 4
GRAD_ACCUM = 4
print(f'Effective batch: {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
import re

def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Training data: same noisy training set (unavoidable — only clean data is the gold test set)
df_tr = pd.read_csv(RESULTS_DIR / 'train_split.csv')

# V2 FIX: Use the clean 300-sample dev set (from gold test) for checkpoint selection
# This prevents the transformer from memorising labelling errors during training
df_dev = pd.read_csv(CLEAN_DEV_CSV)

# V2 FIX: Evaluate on the remaining 1,700 clean test samples (not the full 2,000)
df_test = pd.read_csv(CLEAN_TEST_CSV)

for df in [df_tr, df_dev, df_test]:
    df['text_clean'] = df['text'].apply(preprocess)
    df['label_id'] = df['label'].map(LABEL2ID)

print(f'Train: {len(df_tr):,} (noisy) | Dev (clean): {len(df_dev):,} | Test (clean): {len(df_test):,}')
print('NOTE: Dev and Test are drawn from the 2,000-sample gold-annotated set (stratified split).')
print(f'Label distribution — Dev: {df_dev["label"].value_counts().to_dict()}')
print(f'Label distribution — Test: {df_test["label"].value_counts().to_dict()}')

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': macro_f1}

def make_hf_dataset(df, tokenizer, max_length):
    ds = Dataset.from_dict({
        'text': df['text_clean'].tolist(),
        'label': df['label_id'].tolist(),
    })
    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=max_length)
    ds = ds.map(tokenize, batched=True, batch_size=256)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

def evaluate_on_test(model, tokenizer, df_test, max_length, model_key, results_dir):
    model.eval()
    model.to(DEVICE)
    ds_test = make_hf_dataset(df_test, tokenizer, max_length)
    all_preds = []
    from torch.utils.data import DataLoader
    loader = DataLoader(ds_test, batch_size=64)
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    label_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
    true_labels = df_test['label'].tolist()
    pred_labels = [ID2LABEL[p] for p in all_preds]
    macro_f1 = f1_score(true_labels, pred_labels, average='macro', labels=label_order)
    report_dict = classification_report(true_labels, pred_labels, labels=label_order, target_names=label_order, output_dict=True)
    print(f'\n=== {MODELS_V2[model_key]["name"]} (v2) — Clean Test Set (n={len(df_test):,}) ===')
    print(f'Macro-F1: {macro_f1:.4f}')
    print(classification_report(true_labels, pred_labels, labels=label_order, target_names=label_order))
    cm = confusion_matrix(true_labels, pred_labels, labels=label_order)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order).plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(f'{MODELS_V2[model_key]["name"]} (v2) — Clean Test Set\n(Macro-F1 = {macro_f1:.4f}, n={len(df_test):,})', fontsize=11)
    plt.tight_layout()
    plt.savefig(results_dir / f'{model_key}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    results = {
        'model': MODELS_V2[model_key]['name'],
        'checkpoint': MODELS_V2[model_key]['checkpoint'],
        'macro_f1': round(macro_f1, 4),
        'test_n': len(df_test),
        'per_class': {
            cls: {
                'precision': round(report_dict[cls]['precision'], 4),
                'recall': round(report_dict[cls]['recall'], 4),
                'f1': round(report_dict[cls]['f1-score'], 4),
            } for cls in label_order
        }
    }
    with open(results_dir / f'{model_key}_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    return results

print('Helpers defined.')

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

class DifferentialLRTrainer(Trainer):
    """
    Custom Trainer with differential learning rates:
    - Classification head and pooler: full learning_rate (2e-5)
    - Transformer backbone: 0.25x learning_rate (5e-6)
    
    Motivation: The classification head is randomly initialised and needs aggressive
    updates; the backbone has strong pre-trained representations that should be
    nudged gently to preserve multilingual generalisation.
    """
    def create_optimizer(self):
        no_decay = {"bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight"}
        head_keywords = {"classifier", "pooler"}
        
        head_decay, head_no_decay = [], []
        backbone_decay, backbone_no_decay = [], []
        
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            is_head = any(kw in name for kw in head_keywords)
            no_decay_param = any(nd in name for nd in no_decay)
            
            if is_head:
                if no_decay_param:
                    head_no_decay.append(param)
                else:
                    head_decay.append(param)
            else:
                if no_decay_param:
                    backbone_no_decay.append(param)
                else:
                    backbone_decay.append(param)
        
        backbone_lr = self.args.learning_rate * 0.25  # 5e-6 when base LR = 2e-5
        optimizer_grouped_parameters = [
            {"params": head_decay,        "lr": self.args.learning_rate, "weight_decay": self.args.weight_decay},
            {"params": head_no_decay,     "lr": self.args.learning_rate, "weight_decay": 0.0},
            {"params": backbone_decay,    "lr": backbone_lr,             "weight_decay": self.args.weight_decay},
            {"params": backbone_no_decay, "lr": backbone_lr,             "weight_decay": 0.0},
        ]
        # Filter out empty groups to avoid AdamW errors
        optimizer_grouped_parameters = [g for g in optimizer_grouped_parameters if len(g["params"]) > 0]
        
        # Use adam_epsilon from TrainingArguments so callers can override per-model
        # (mDeBERTa requires eps=1e-6 per the DeBERTa paper; others use default 1e-8)
        adam_eps = getattr(self.args, 'adam_epsilon', 1e-8)
        self.optimizer = AdamW(optimizer_grouped_parameters, eps=adam_eps)
        return self.optimizer

print('DifferentialLRTrainer defined.')
print('  Head LR: 2e-5 | Backbone LR: 5e-6')

In [ ]:
def reinit_mdeberta_classifier(model):
    """
    Re-initialise ALL Linear layers in mDeBERTa-v3's classification head.

    mDeBERTa-v3 (DebertaV2ForSequenceClassification) has a TWO-component head:
      model.pooler     — ContextPooler with a dense Linear(hidden, pooler_hidden)
      model.classifier — Linear(pooler_hidden, num_labels)

    CRITICAL: BOTH must be re-initialised. The original fix only targeted
    model.classifier and missed model.pooler. With pooler uninitialised,
    its Kaiming-uniform weights amplify activations to extreme magnitudes
    (observed: loss = 38,642 at step 50), causing FP32 overflow on the first
    backward pass. All weights then become NaN and are never recovered.
    """
    components = []
    if hasattr(model, 'pooler'):
        components.append(('pooler', model.pooler))
    if hasattr(model, 'classifier'):
        components.append(('classifier', model.classifier))

    reinit_count = 0
    for name, component in components:
        for module in component.modules():
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()
                reinit_count += 1
                print(f'  Reinit: {name} -> {module}')

    print(f'  Total Linear layers re-initialised: {reinit_count}')
    assert reinit_count >= 2, (
        f'Expected >=2 Linear layers (pooler.dense + classifier), got {reinit_count}. '
        'Check model architecture.'
    )
    return reinit_count

print('reinit_mdeberta_classifier() defined (v2 — reinits pooler + classifier).')


In [ ]:
def finetune_model_v2(model_key: str) -> dict:
    """
    Fine-tune a single model with all v2 fixes applied:
    1. Clean dev set for checkpoint selection (not noisy auto-labels)
    2. 7 epochs (was 3)
    3. Gradient clipping max_norm=1.0
    4. mDeBERTa: FP16=False, BF16=False (both disabled), plus classifier re-init
    5. Differential LR: head=2e-5, backbone=5e-6
    6. mDeBERTa: adam_epsilon=1e-6 (DeBERTa paper recommendation)
    7. mDeBERTa: lower LR (5e-6 peak) + minimal warmup (5%)
       — v2 training showed the model learns well at LR≈3.8e-6 (step 150, loss=0.66/sample)
         but gets pushed off that solution when warmup ramps LR to 2e-5 at step 250.
         Capping peak LR at 5e-6 stays in the learning regime and avoids the overshoot.
    """
    cfg = MODELS_V2[model_key]
    checkpoint = cfg['checkpoint']
    results_dir = cfg['results_dir']
    results_dir.mkdir(parents=True, exist_ok=True)
    output_dir = results_dir / 'checkpoints'

    print(f'\n{"="*65}')
    print(f'MODEL (v2): {cfg["name"]}')
    print(f'Checkpoint: {checkpoint}')
    print(f'{"="*65}')
    print(f'V2 fixes: clean dev ({len(df_dev)} samples), {FINETUNE_CONFIG_V2["num_train_epochs"]} epochs, '
          f'grad_clip={FINETUNE_CONFIG_V2["max_grad_norm"]}, diff_LR')

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    max_length = FINETUNE_CONFIG_V2['max_seq_length']

    print('Tokenising splits...')
    ds_train = make_hf_dataset(df_tr,  tokenizer, max_length)
    ds_dev   = make_hf_dataset(df_dev, tokenizer, max_length)
    print(f'  Train: {len(ds_train):,}  |  Clean Dev: {len(ds_dev):,}')

    print('Loading model...')
    # NOTE: do NOT pass ignore_mismatched_sizes=True for base models being loaded
    # as classifiers. For a base checkpoint (no existing classifier/pooler),
    # the new head layers are simply absent from the checkpoint and are auto-
    # initialised — no size mismatch error occurs. Using ignore_mismatched_sizes=True
    # is dangerous: it silently re-initialises ANY mismatched layer, including
    # backbone layers such as rel_embeddings that differ between transformers versions.
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        # ignore_mismatched_sizes intentionally omitted — see comment above
    )

    if model_key == 'mdeberta':
        # ── mDeBERTa-specific overrides ────────────────────────────────────────
        use_fp16   = False
        use_bf16   = False       # CRITICAL: NOT torch.cuda.is_bf16_supported()
        adam_eps   = 1e-6        # DeBERTa paper: larger epsilon for Adam stability
        # LR override: training history showed the model learns well at LR≈3.8e-6
        # (loss dropped to 0.66/sample at step 150) but the 20%-warmup ramp pushed LR
        # to 6.4e-6 by step 250, overshooting the good solution and collapsing to
        # all-NEUTRAL for the remaining 3,656 steps. Capping peak LR at 5e-6 with
        # 5% warmup keeps the model in the stable learning regime throughout.
        peak_lr    = 5e-6        # was 2e-5 — caused overshoot at warmup peak
        warmup_r   = 0.05        # was 0.20 — 20% warmup ramped past the stable LR zone
        print(f'  mDeBERTa overrides: peak_lr={peak_lr:.0e}, warmup={warmup_r}, '
              f'fp16=False, bf16=False, adam_eps={adam_eps:.0e}')
        reinit_mdeberta_classifier(model)

        # ── Pre-training sanity check ──────────────────────────────────────────
        # Verify backbone hidden states are not extreme before wasting GPU time.
        print('\n  ── Pre-training sanity check ──')
        model.eval()
        model.to(DEVICE)
        _sample = ds_train[:4]
        with torch.no_grad():
            _out = model(
                input_ids      = _sample['input_ids'].to(DEVICE),
                attention_mask = _sample['attention_mask'].to(DEVICE),
            )
        _logits = _out.logits.float()
        _labels = _sample['label'].to(DEVICE)
        _loss   = F.cross_entropy(_logits, _labels).item()
        print(f'  Logit stats  — mean: {_logits.mean():.4f}  std: {_logits.std():.4f}  max_abs: {_logits.abs().max():.4f}')
        print(f'  CE loss (4 samples): {_loss:.4f}  (expected ≈ {torch.log(torch.tensor(float(NUM_LABELS))):.4f} for random init)')
        if _logits.abs().max() > 50:
            raise RuntimeError(
                f'FATAL: max |logit| = {_logits.abs().max():.1f} >> 50 — backbone is producing '
                'extreme activations. The model checkpoint likely has a mismatched layer '
                'that was silently re-initialised. Check transformers version compatibility '
                'with microsoft/mdeberta-v3-base and inspect loading warnings above.'
            )
        if _loss > 5.0:
            print(f'  WARNING: initial CE loss = {_loss:.4f} is high (expected ~1.10). '
                  'Training may be slow to converge. Proceeding anyway.')
        else:
            print(f'  ✓ Sanity check passed — logits and initial loss are in normal range.')
        model.train()
        print()
    else:
        use_fp16 = (DEVICE == 'cuda')
        use_bf16 = (DEVICE == 'mps')
        adam_eps = 1e-8
        peak_lr  = FINETUNE_CONFIG_V2['learning_rate']
        warmup_r = FINETUNE_CONFIG_V2['warmup_ratio']

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=FINETUNE_CONFIG_V2['num_train_epochs'],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=FINETUNE_CONFIG_V2['per_device_eval_batch_size'],
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=peak_lr,                               # model-specific peak LR
        warmup_ratio=warmup_r,                               # model-specific warmup
        weight_decay=FINETUNE_CONFIG_V2['weight_decay'],
        max_grad_norm=FINETUNE_CONFIG_V2['max_grad_norm'],
        adam_epsilon=adam_eps,                               # model-specific epsilon
        eval_strategy=FINETUNE_CONFIG_V2['eval_strategy'],
        save_strategy=FINETUNE_CONFIG_V2['save_strategy'],
        load_best_model_at_end=FINETUNE_CONFIG_V2['load_best_model_at_end'],
        metric_for_best_model=FINETUNE_CONFIG_V2['metric_for_best_model'],
        greater_is_better=True,
        logging_steps=FINETUNE_CONFIG_V2['logging_steps'],
        fp16=use_fp16,
        bf16=use_bf16,
        dataloader_num_workers=FINETUNE_CONFIG_V2['dataloader_num_workers'],
        seed=SEED,
        report_to='none',
        save_total_limit=2,
    )

    trainer = DifferentialLRTrainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_dev,
        compute_metrics=compute_metrics,
    )

    print(f'Starting v2 fine-tuning on {DEVICE} ({FINETUNE_CONFIG_V2["num_train_epochs"]} epochs)...')
    trainer.train()
    print(f'Best checkpoint: {trainer.state.best_model_checkpoint}')
    print(f'Best clean-dev Macro-F1: {trainer.state.best_metric:.4f}')

    history = trainer.state.log_history
    with open(results_dir / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    results = evaluate_on_test(trainer.model, tokenizer, df_test, max_length, model_key, results_dir)

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

print('finetune_model_v2() defined.')

## XLM-R v2

XLM-RoBERTa base (`xlm-roberta-base`, ~278M params). General-purpose multilingual transformer trained on CommonCrawl in 100 languages. Architectural baseline — no domain-specific pretraining.

In [ ]:
xlm_r_v2_results = finetune_model_v2('xlm_r')

## mDeBERTa-v3 v2

`microsoft/mdeberta-v3-base` (~278M params). Stronger general-architecture baseline using disentangled attention and enhanced mask decoder. In v1 this model produced **zero weight updates** due to the combined effect of BF16 precision and extreme initialisation logits. Both are fixed here.

In [ ]:
mdeberta_v2_results = finetune_model_v2('mdeberta')

## XLM-T v2

`cardiffnlp/twitter-xlm-roberta-base-sentiment` (~278M params). Domain-adapted model: XLM-R further pretrained on ~198M tweets, then fine-tuned for sentiment classification. Twitter pretraining and prior sentiment fine-tuning are named confounds — gains over XLM-R cannot be attributed to architecture alone.

In [ ]:
xlm_t_v2_results = finetune_model_v2('xlm_t')

## V1 vs V2 Comparison

Loads saved v1 results (if present) and compares Macro-F1 scores.  

**Caveat:** V1 evaluated on the full 2,000-sample gold test set; V2 evaluates on 1,700 samples (the 300 used for dev are withheld). Direct numeric comparison is approximate — the methodological improvement is the key result.

In [ ]:
# V1 vs V2 comparison — loads all results from disk so this cell works
# even if XLM-R / XLM-T training cells were not run in this session.
import json

def load_json_safe(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None

# V1 results (original flawed run)
xlm_r_v1    = load_json_safe(RESULTS_DIR / 'xlm_r'      / 'xlm_r_results.json')
xlm_t_v1    = load_json_safe(RESULTS_DIR / 'xlm_t'      / 'xlm_t_results.json')

# V2 results — loaded from disk, not from in-memory variables
xlm_r_v2    = load_json_safe(RESULTS_DIR / 'xlm_r_v2'   / 'xlm_r_results.json')
mdeberta_v2 = load_json_safe(RESULTS_DIR / 'mdeberta_v2' / 'mdeberta_results.json')
xlm_t_v2    = load_json_safe(RESULTS_DIR / 'xlm_t_v2'   / 'xlm_t_results.json')

print('\n=== V1 vs V2 COMPARISON (Macro-F1) ===')
print(f'{"Model":<30} {"V1 (noisy dev)":>16} {"V2 (clean dev)":>16} {"Delta":>8}')
print('-' * 74)

for key, v1, v2 in [
    ('XLM-R',    xlm_r_v1,    xlm_r_v2),
    ('XLM-T',    xlm_t_v1,    xlm_t_v2),
    ('mDeBERTa', None,         mdeberta_v2),
]:
    v1_f1 = v1['macro_f1'] if v1 else float('nan')
    v2_f1 = v2['macro_f1'] if v2 else float('nan')
    delta = v2_f1 - v1_f1 if (v1 and v2) else float('nan')
    delta_str = f'{delta:+.4f}' if not (delta != delta) else ('N/A (was failed)' if not v1 else 'pending')
    print(f'{key:<30} {v1_f1:>16.4f} {v2_f1:>16.4f} {delta_str:>8}')

print('\nNote: V1 evaluated on n=2,000 gold test; V2 evaluated on n=1,700 clean test.')
print('Direct score comparison is approximate; methodology comparison is the key takeaway.')
